# Agency Economically Significant Final Rules by Presidential Year
**Author:** Henry Hirsch  
**Date:** 2025-02-13

## Initialize

Set the project root and configure paths to data and output directories. All paths are relative to the project root.

In [ ]:
import os
import shutil
from pathlib import Path
from datetime import date

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.image import imread
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# ---------------------------------------------------------------------------
# Project root — adjust this if running from a different working directory
# ---------------------------------------------------------------------------
# Mirrors R's here::i_am("charts/code/agency_econ_significant_rules_by_presidential_year.Rmd")
# Assumes the notebook sits at <project_root>/charts/code/
PROJECT_ROOT = Path(os.getcwd()).parents[1]   # go up two levels from charts/code/

DATA_DIR   = PROJECT_ROOT / "charts" / "data"
OUTPUT_DIR = PROJECT_ROOT / "charts" / "output" / "by_agency"
SOURCE_DIR = PROJECT_ROOT / "data" / "es_rules"
LOGO_PATH  = PROJECT_ROOT / "charts" / "assets" / "logo.png"   # adjust as needed

DATA_FILE  = "agency_econ_significant_rules_by_presidential_year.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

### Copy dataset (mirrors `copy_dataset()` from `local_utils.R`)

In [ ]:
def copy_dataset(filename: str, src_dir: Path, dst_dir: Path) -> None:
    """Copy *filename* from src_dir to dst_dir, mirroring R's copy_dataset()."""
    src = src_dir / filename
    dst = dst_dir / filename
    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied: {src} → {dst}")
    else:
        print(f"Source file not found: {src}. Using existing file in data dir if present.")

copy_dataset(DATA_FILE, SOURCE_DIR, DATA_DIR)

## Style — mirrors `style.R`

Defines the RSC colour palette, typography, and `theme_RSC` grid style used across all charts.

In [ ]:
# ---------------------------------------------------------------------------
# Colour palette  (mirrors style.R)
# ---------------------------------------------------------------------------
GWblue    = "#003366"   # deep navy — Democratic bars
red       = "#CC0000"   # RSC red   — Republican bars
lightblue = "#6699CC"   # stripe fill for Democratic bars
RSCgray   = "#AAAAAA"   # grid lines & axis lines

PARTY_COLORS   = {"Democratic": GWblue, "Republican": red}
PARTY_HATCHES  = {"Democratic": "////", "Republican": ""}   # hatching = stripe

# ---------------------------------------------------------------------------
# Typography / RC params  (mirrors theme_RSC)
# ---------------------------------------------------------------------------
matplotlib.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": True,
    "axes.edgecolor":     RSCgray,
    "axes.linewidth":     1.0,
    "xtick.bottom":       False,
    "ytick.left":         False,
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
})

## Load Data

In [ ]:
agency_es_rules_NA = pd.read_csv(DATA_DIR / DATA_FILE)

# Rename columns (mirrors colnames() assignment in R)
agency_es_rules_NA.columns = ["name", "acronym", "year", "party", "rules"]

agency_es_rules_NA.head()

## Process Data

In [ ]:
# Drop rows with any NA values  (mirrors complete.cases())
agency_es_rules = agency_es_rules_NA.dropna().copy()

# Rename STATE → DOS
agency_es_rules["acronym"] = agency_es_rules["acronym"].replace("STATE", "DOS")

# Current date for caption
current_date = date.today().strftime("%B %d, %Y")

print(f"Rows after cleaning: {len(agency_es_rules)}")
print(f"Unique agencies:     {sorted(agency_es_rules['acronym'].unique())}")
print(f"Current date:        {current_date}")

## Set Variables

In [ ]:
# Target agencies (lowercase, matching the R vector)
agencies = [
    "dhs", "doc", "dod", "doe", "doi", "doj", "dol", "dos", "dot",
    "ed", "epa",
    "hhs", "hud",
    "sba",
    "treas",
    "usda",
    "va",
]

# Caption text (mirrors paste() + strwrap() in R)
caption_text = (
    "Sources: Office of the Federal Register (federalregister.gov) for the years 2021 and onwards;\n"
    " Office of Information and Regulatory Affairs (reginfo.gov) for all the prior years.\n\n"
    f"Updated: {current_date}"
)

## Graphing Functions

### Helper: Y-axis interval

In [ ]:
def get_interval(values) -> int:
    """Return a round y-axis tick interval, mirroring R's get_interval()."""
    max_val = int(max(values))
    if max_val <= 20:
        interval = round(max_val / 5)
    else:
        interval = max(int(max_val / 10) * 5, 5)
    return max(interval, 1)   # guard against 0


def ydynam(df, interval: int, padding: int = 5) -> int:
    """Compute upper y-axis limit (mirrors R's ydynam())."""
    max_val = int(df["rules"].max())
    upper = (int(max_val / interval) + 1) * interval + padding
    return upper

### Main plotting function

In [ ]:
def plot_data(df: pd.DataFrame, agency_acronym: str, agency_name) -> plt.Figure:
    """
    Replicate the ggplot bar chart with diagonal-stripe hatching for Democratic
    bars and solid fill for Republican bars.

    Parameters
    ----------
    df             : filtered dataframe for a single agency
    agency_acronym : e.g. 'epa'
    agency_name    : full agency name (may be a Series; first value is used)
    """
    if hasattr(agency_name, "iloc"):
        agency_name = agency_name.iloc[0]

    interval = get_interval(df["rules"])
    upper    = ydynam(df, interval, padding=5)

    years  = sorted(df["year"].unique())
    parties = df.drop_duplicates("year").set_index("year")["party"]

    fig, ax = plt.subplots(figsize=(12, 9))

    bar_width = 0.5

    for yr in years:
        row   = df[df["year"] == yr].iloc[0]
        party = row["party"]
        rules = row["rules"]
        color  = PARTY_COLORS.get(party, GWblue)
        hatch  = PARTY_HATCHES.get(party, "")

        bar = ax.bar(
            yr, rules,
            width      = bar_width,
            color      = color,
            hatch      = hatch,
            edgecolor  = lightblue if hatch else color,
            linewidth  = 0,
        )

        # Overlay hatching in lightblue on top of the navy fill for Democratic bars
        if hatch:
            ax.bar(
                yr, rules,
                width      = bar_width,
                color      = "none",
                hatch      = hatch,
                edgecolor  = lightblue,
                linewidth  = 0,
            )

    # ---------------------------------------------------------------------------
    # Axes formatting  (mirrors theme_RSC + per-plot theme() overrides)
    # ---------------------------------------------------------------------------
    ax.set_title(
        f"{agency_acronym.upper()} Economically Significant Final Rules\nPublished by Presidential Year",
        fontsize=14, fontweight="bold", ha="center", pad=25
    )
    ax.set_ylabel("Number of Rules", fontsize=11)
    ax.set_xlabel("")

    # X axis
    ax.set_xlim(min(years) - 0.5, max(years) + 0.5)
    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=65, ha="right", va="top", fontsize=9)
    ax.tick_params(axis="x", length=0)          # no tick marks (axis.ticks.x = element_blank)
    ax.spines["bottom"].set_linewidth(1)
    ax.spines["bottom"].set_color(RSCgray)

    # Y axis
    ax.set_ylim(0, upper)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(interval))
    ax.tick_params(axis="y", length=0)

    # Grid  (panel.grid.major.y = line, panel.grid.major.x = blank)
    ax.yaxis.grid(True, color=RSCgray, linestyle="-", linewidth=0.8)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)

    # No legend (legend.position = "none")
    ax.legend_.remove() if ax.get_legend() else None

    # Caption
    fig.text(
        0.98, 0.01, caption_text,
        ha="right", va="bottom",
        fontsize=8, color="#555555",
        transform=fig.transFigure,
    )

    # ---------------------------------------------------------------------------
    # Logo overlay  (mirrors draw_image() in ggdraw)
    # ---------------------------------------------------------------------------
    if LOGO_PATH.exists():
        logo_img = imread(str(LOGO_PATH))
        newax = fig.add_axes([0.09, 0.079, 0.14, 0.10])   # [left, bottom, width, height]
        newax.imshow(logo_img)
        newax.axis("off")

    plt.tight_layout(rect=[0, 0.06, 1, 1])
    return fig

### Save function

In [ ]:
def save_plot_png_pdf(fig: plt.Figure, save_path: Path, agency_acronym: str) -> None:
    """Save figure as both PDF and PNG, mirroring save_plot_png_pdf() in R."""
    image_name = f"{agency_acronym.lower()}_econ_significant_rules_by_presidential_year"

    # PDF
    pdf_path = save_path / f"{image_name}.pdf"
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight", facecolor="white")

    # PNG  (1200 × 900 px @ 96 dpi  ≡ 12.5 × 9.375 in, close to R's 1200×900 px/96dpi)
    png_path = save_path / f"{image_name}.png"
    fig.savefig(png_path, dpi=96, bbox_inches="tight", facecolor="white")

    print(f"  Saved: {pdf_path.name}")
    print(f"  Saved: {png_path.name}")

## Pipeline

Iterates over each target agency, filters the data, builds the chart, and saves both PNG and PDF outputs — exactly as the `for` loop in the Rmd does.

In [ ]:
agency_names_list = {}

for agency_a in agencies:
    df_agency = agency_es_rules[agency_es_rules["acronym"] == agency_a.upper()]

    if df_agency.empty:
        print(f"[SKIP] {agency_a.upper()} — no data found")
        continue

    agency_n = df_agency["name"]
    agency_names_list[agency_a.upper()] = agency_n.iloc[0]

    print(f"\n[{agency_a.upper()}] {agency_n.iloc[0]}")

    fig = plot_data(df_agency, agency_a, agency_n)
    save_plot_png_pdf(fig, OUTPUT_DIR, agency_a)
    plt.show()
    plt.close(fig)

print("\nDone.")